# Evaluation Benchmarks with alignrl

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sacredvoid/alignrl/blob/main/notebooks/04_evaluation_benchmarks.ipynb)

This notebook demonstrates **rigorous evaluation** across training stages using standardized benchmarks. Evaluation is arguably the most important part of post-training: without it, you can't tell if your SFT, GRPO, or DPO actually improved the model, or if you introduced regressions on other capabilities. We use the `lm-evaluation-harness` under the hood, wrapped in alignrl's evaluation API for convenience.

## Setup

Install alignrl with evaluation dependencies.

In [ ]:
!pip install alignrl[train,unsloth,eval] -q

## Why Rigorous Evaluation Matters

Post-training can improve one capability while degrading others. Common failure modes:

- **Alignment tax**: DPO improves helpfulness but hurts factual accuracy
- **Catastrophic forgetting**: SFT on a narrow dataset breaks general capabilities
- **Reward hacking**: GRPO finds shortcuts that game the reward without actually solving problems

The solution is to evaluate on **multiple benchmarks** at **every training stage** and compare the results side by side.

### Benchmarks We Use

| Benchmark | What It Tests | Format |
|-----------|--------------|--------|
| GSM8K | Grade-school math reasoning | Chain-of-thought word problems |
| ARC-Challenge | Science reasoning | Multiple-choice science questions |

## Configuration

We configure the evaluator to run GSM8K and ARC-Challenge with a small `limit` to keep runtime manageable on Colab. In production, you'd remove the limit for full benchmark scores.

In [ ]:
from alignrl.eval import EvalConfig, EvalRunner, compare_stages

config = EvalConfig(
    model_name="Qwen/Qwen2.5-3B",
    tasks=["gsm8k", "arc_challenge"],
    num_fewshot=0,
    batch_size="auto",
    limit=50,  # Evaluate on 50 examples per benchmark (remove for full eval)
    load_in_4bit=True,
)

print(f"Model: {config.model_name}")
print(f"Tasks: {config.tasks}")
print(f"Limit: {config.limit} examples per task")

## Evaluate the Base Model

First, let's establish a baseline by evaluating the untrained model.

In [ ]:
runner = EvalRunner(config)

print("Evaluating base model...")
base_result = runner.evaluate(stage="base")

print(f"\nBase model results:")
for benchmark, metrics in base_result.benchmarks.items():
    print(f"  {benchmark}:")
    for metric, score in metrics.items():
        if isinstance(score, float):
            print(f"    {metric}: {score:.4f}")

## Evaluate All Training Stages

If you've run the previous notebooks and have adapter weights saved, we can evaluate each stage. If not, we'll just evaluate the base model and show how the comparison would work.

The `evaluate_all_stages` method takes a dictionary of `{stage_name: adapter_path}` where `None` means the base model.

In [ ]:
from pathlib import Path

# Define adapter paths (None = base model, no adapter)
adapter_stages = {"base": None}

# Check which adapters are available from previous notebooks
for stage, path in [("sft", "./outputs/sft/final"), ("grpo", "./outputs/grpo/final"), ("dpo", "./outputs/dpo/final")]:
    if Path(path).exists():
        adapter_stages[stage] = path
        print(f"Found {stage} adapter at {path}")
    else:
        print(f"No {stage} adapter found at {path} (run notebook {['01', '02', '03'][['sft', 'grpo', 'dpo'].index(stage)]} first)")

print(f"\nEvaluating stages: {list(adapter_stages.keys())}")

In [ ]:
# Run evaluation across all available stages
all_results = runner.evaluate_all_stages(adapter_stages)

for result in all_results:
    print(f"\n--- Stage: {result.stage} ---")
    for benchmark, metrics in result.benchmarks.items():
        print(f"  {benchmark}:")
        for metric, score in metrics.items():
            if isinstance(score, float):
                print(f"    {metric}: {score:.4f}")

## Comparison Table

The `compare_stages` function reorganizes results into a benchmark-centric view for easy comparison.

In [ ]:
comparison = compare_stages(all_results)

# Build a comparison table
stages = list(adapter_stages.keys())

print(f"{'Benchmark':<20} {'Metric':<25}", end="")
for stage in stages:
    print(f"{stage:>12}", end="")
print()
print("-" * (45 + 12 * len(stages)))

for benchmark, stage_metrics in comparison.items():
    # Get all metrics from any stage
    all_metrics = set()
    for metrics in stage_metrics.values():
        all_metrics.update(metrics.keys())

    for metric in sorted(all_metrics):
        print(f"{benchmark:<20} {metric:<25}", end="")
        for stage in stages:
            score = stage_metrics.get(stage, {}).get(metric, float('nan'))
            if isinstance(score, float):
                print(f"{score:>12.4f}", end="")
            else:
                print(f"{'N/A':>12}", end="")
        print()

## Radar Chart Visualization

A radar chart makes it easy to see at a glance where each stage excels or falls behind.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Collect the primary accuracy metric for each benchmark per stage
benchmark_names = list(comparison.keys())
stage_scores = {}

for stage in stages:
    scores = []
    for benchmark in benchmark_names:
        metrics = comparison.get(benchmark, {}).get(stage, {})
        # Pick the first float metric as the primary score
        primary = 0.0
        for k, v in metrics.items():
            if isinstance(v, float) and ("acc" in k or "exact_match" in k):
                primary = v
                break
        scores.append(primary)
    stage_scores[stage] = scores

# Create radar chart
N = len(benchmark_names)
if N >= 2:
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]  # Close the polygon

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

    colors = {"base": "#6b7280", "sft": "#2563eb", "grpo": "#16a34a", "dpo": "#9333ea"}

    for stage, scores in stage_scores.items():
        values = scores + scores[:1]
        ax.plot(angles, values, "o-", linewidth=2, label=stage, color=colors.get(stage, "#000"))
        ax.fill(angles, values, alpha=0.1, color=colors.get(stage, "#000"))

    ax.set_thetagrids(np.degrees(angles[:-1]), benchmark_names)
    ax.set_ylim(0, 1)
    ax.set_title("Benchmark Comparison Across Training Stages", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.show()
else:
    print("Need at least 2 benchmarks for a radar chart. Showing bar chart instead.")
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(N)
    width = 0.8 / len(stages)
    for i, (stage, scores) in enumerate(stage_scores.items()):
        ax.bar(x + i * width, scores, width, label=stage, color=colors.get(stage, "#000"))
    ax.set_xticks(x + width * (len(stages) - 1) / 2)
    ax.set_xticklabels(benchmark_names)
    ax.set_ylabel("Score")
    ax.set_title("Benchmark Comparison")
    ax.legend()
    plt.tight_layout()
    plt.show()

## Save Results for GitHub Pages

The alignrl results dashboard reads JSON files to display evaluation results. Let's save them in the expected format.

In [ ]:
import json

output_dir = Path("./results")
runner.save_results(all_results, output_dir)

print("Saved evaluation results:")
for f in sorted(output_dir.rglob("*.json")):
    print(f"  {f}")

# Show the comparison JSON
with open(output_dir / "comparison.json") as f:
    print("\ncomparison.json:")
    print(json.dumps(json.load(f), indent=2))

## Interpreting Results

When analyzing your evaluation results, look for:

1. **Consistent improvement**: Does each training stage improve on the previous one for the target task?
2. **Regression checks**: Did any stage significantly hurt performance on non-target benchmarks?
3. **Diminishing returns**: How much does each stage contribute? Is DPO worth it after GRPO?

For a 3B model trained on small subsets (as in this notebook series), don't expect dramatic improvements. The goal is to demonstrate the workflow and verify that nothing broke.

## Next Steps

- **Notebook 05 (Inference)**: Serve the best-performing stage with optimized inference backends
- **Full evaluation**: Remove the `limit=50` parameter and run on full benchmarks for publishable results
- **More benchmarks**: Add MMLU, HellaSwag, TruthfulQA for broader coverage